# 07 · Análisis estadístico
**Responsables:** P5 (librería e hipótesis principal), P2 (hipótesis A), P4 (hipótesis B) · **Código:** `src/estadistica.py` · **Salidas:** `outputs/stats/`

**Plan fijado antes de ver los resultados** (no se cambia según lo que salga)
- α = 0,05. Hipótesis principal unilateral: H1 = asociación **positiva** entre `prop_18_44_pct` y `puntaje_mdm` (2020, hasta 123 municipios).
- Pearson y Spearman con IC 95 % y valor p. Spearman es la principal si Shapiro-Wilk rechaza normalidad o si hay influyentes.
- Regresión simple y con controles: log de la población, % rural y grupo de capacidades del MDM (G1–G6). Breusch-Pagan → errores HC3 si hay heterocedasticidad. Distancia de Cook > 4/n para influyentes. VIF.
- Sensibilidad: sin influyentes, con el bloque alterno del MDM y con la proporción calculada desde la fuente por áreas.
- Complementarias (exploratorias, p ajustado por Holm): A = producción vs población 18–64; B = % de suelo III–IV vs rendimiento de papa; D = mujeres y hombres 18–44 vs desempeño.

**Cómo concluir:** "hay evidencia de asociación positiva", "no se rechaza H0" o "datos insuficientes", siempre con la magnitud (r) y su intervalo. **Nunca** causalidad ni "disposición al cambio".

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [ ]:
import pandas as pd
from src import estadistica as est, graficos
from src.silver_poblacion import prop_18_44_desde_areas

ma = pl.read_parquet(GOLD / "municipio_anio.parquet")
d2020 = ma.filter(pl.col("anio") == ANIO_MDM).to_pandas()
d2023 = ma.filter(pl.col("anio") == ANIO_EVA).to_pandas()

mdm = pl.read_parquet(SILVER / "desempeno.parquet")
alt_mdm = mdm.filter(~pl.col("es_bloque_oficial")).select("codigo_municipio", puntaje_mdm_alt="puntaje_mdm")
alt_pob = prop_18_44_desde_areas(pl.read_parquet(SILVER / "poblacion_area.parquet")).filter(pl.col("anio") == ANIO_MDM)
alterno = alt_mdm.join(alt_pob.drop("anio"), on="codigo_municipio", how="full", coalesce=True).to_pandas()

## Hipótesis principal (P5)

In [ ]:
principal = est.hipotesis_principal(d2020, alterno)
principal[["analisis", "metodo", "n", "r", "ic95_inf", "ic95_sup", "p", "conclusion"]]

In [ ]:
pd.read_csv(STATS / "h_principal_normalidad.csv")

In [ ]:
print((STATS / "regresion_h_principal_controles_resumen.txt").read_text(encoding="utf-8"))

## Hipótesis complementarias (P2, P4) y consolidado con Holm

In [ ]:
a = est.hipotesis_produccion_poblacion(d2023)   # P2
b = est.hipotesis_suelo_rendimiento(d2023)      # P4
d = est.hipotesis_sexo(d2020)                   # P5
todo = est.consolidar(principal, [a, b, d])
todo[["tipo", "analisis", "metodo", "n", "r", "p", "p_holm", "conclusion"]]

## Dispersiones (P5)

In [ ]:
fila = principal[(principal.analisis == "principal") & (principal.metodo == "spearman")].iloc[0].to_dict()
graficos.dispersion(d2020, "prop_18_44_pct", "puntaje_mdm",
                    f"Población de 18-44 años vs desempeño municipal, {ANIO_MDM}",
                    "Población de 18 a 44 años (% del total)", "Puntaje MDM (0-100)",
                    "DANE, proyecciones; DNP, Medición del Desempeño Municipal", "p5_dispersion_joven_mdm", fila)
fa = a.iloc[0].to_dict()
graficos.dispersion(d2023, "pob_18_64", "produccion_total_t",
                    f"Producción agrícola vs población de 18-64 años, {ANIO_EVA}",
                    "Población de 18 a 64 años (personas, escala log)", "Producción total (t, escala log)",
                    "UPRA - EVA; DANE - proyecciones", "p5_dispersion_produccion_poblacion", fa, log=True)